# 03 - Tolerance Analysis

This notebook analyzes variance in recordings to determine appropriate tolerance
multipliers for sign matching.

## Contents
1. Analyze variance in recordings
2. Determine tolerance multipliers
3. Visualize tolerance regions
4. Test matching with different thresholds

In [ ]:
# Common imports
import sys
sys.path.insert(0, '..')

from src.types import *
from src.normalize import *
from src.landmarks import *
from src.validation import *

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from pathlib import Path
from scipy import stats

# Local visualization utilities
from notebook_utils import (
    plot_hand_3d,
    plot_hand_2d,
    plot_tolerance_region,
    plot_comparison_overlay,
    plot_variance_heatmap,
    plot_distance_distribution,
    landmarks_to_arrays,
    set_notebook_style,
    create_sample_landmarks,
    FINGERTIP_INDICES,
)

set_notebook_style()
%matplotlib inline

print("Imports loaded successfully!")

In [ ]:
# Generate synthetic samples with known variance for testing
def generate_samples_with_noise(base_landmarks, n_samples=20, noise_std=0.02):
    """
    Generate samples by adding Gaussian noise to base landmarks.
    
    Args:
        base_landmarks: Base landmark positions
        n_samples: Number of samples to generate
        noise_std: Standard deviation of noise
    
    Returns:
        List of landmark sets
    """
    x, y, z = landmarks_to_arrays(base_landmarks)
    base_coords = np.stack([x, y, z], axis=1)
    
    samples = []
    for _ in range(n_samples):
        noise = np.random.normal(0, noise_std, base_coords.shape)
        noisy_coords = base_coords + noise
        sample = [Point3D(x=noisy_coords[i, 0], y=noisy_coords[i, 1], z=noisy_coords[i, 2]) 
                  for i in range(len(base_landmarks))]
        samples.append(sample)
    
    return samples

# Create base hand and samples
base_hand = create_sample_landmarks()
samples_low_var = generate_samples_with_noise(base_hand, n_samples=30, noise_std=0.01)
samples_med_var = generate_samples_with_noise(base_hand, n_samples=30, noise_std=0.03)
samples_high_var = generate_samples_with_noise(base_hand, n_samples=30, noise_std=0.05)

print(f"Generated samples with different variance levels:")
print(f"  Low variance (std=0.01): {len(samples_low_var)} samples")
print(f"  Medium variance (std=0.03): {len(samples_med_var)} samples")
print(f"  High variance (std=0.05): {len(samples_high_var)} samples")

## 1. Analyze Variance in Recordings

In [ ]:
# Calculate variance statistics
def analyze_variance(samples):
    """
    Analyze variance statistics across samples.
    
    Returns:
        Dict with variance metrics
    """
    # Stack all samples
    coords_list = []
    for s in samples:
        x, y, z = landmarks_to_arrays(s)
        coords_list.append(np.stack([x, y, z], axis=1))
    
    stacked = np.stack(coords_list, axis=0)  # (n_samples, n_landmarks, 3)
    
    # Calculate statistics
    mean = np.mean(stacked, axis=0)
    var = np.var(stacked, axis=0)
    std = np.std(stacked, axis=0)
    
    # Per-landmark total variance
    landmark_var = np.sum(var, axis=1)  # Sum across x, y, z
    
    # Distances from mean
    distances = np.sqrt(np.sum((stacked - mean) ** 2, axis=2))  # (n_samples, n_landmarks)
    
    return {
        'mean': mean,
        'variance': var,
        'std': std,
        'landmark_variance': landmark_var,
        'distances': distances,
        'mean_distance': np.mean(distances),
        'max_distance': np.max(distances),
        'percentile_95': np.percentile(distances.flatten(), 95),
        'percentile_99': np.percentile(distances.flatten(), 99),
    }

# Analyze each variance level
analysis = {
    'low': analyze_variance(samples_low_var),
    'medium': analyze_variance(samples_med_var),
    'high': analyze_variance(samples_high_var),
}

print("Variance Analysis Results:")
print("="*60)
for level, stats in analysis.items():
    print(f"\n{level.upper()} VARIANCE:")
    print(f"  Mean distance from canonical: {stats['mean_distance']:.4f}")
    print(f"  Max distance: {stats['max_distance']:.4f}")
    print(f"  95th percentile: {stats['percentile_95']:.4f}")
    print(f"  99th percentile: {stats['percentile_99']:.4f}")

In [ ]:
# Visualize variance by landmark
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, (level, stats) in zip(axes, analysis.items()):
    landmark_var = stats['landmark_variance']
    
    # Color by finger
    colors = ['purple'] + ['red']*4 + ['green']*4 + ['blue']*4 + ['orange']*4 + ['yellow']*4
    
    ax.bar(range(21), landmark_var, color=colors)
    ax.set_xlabel('Landmark Index')
    ax.set_ylabel('Total Variance')
    ax.set_title(f'{level.title()} Variance')
    
    # Highlight fingertips
    for idx in FINGERTIP_INDICES:
        ax.axvline(idx, color='black', linestyle=':', alpha=0.3)

plt.tight_layout()
plt.show()

print("Note: Fingertips typically have higher variance than palm landmarks")

In [ ]:
# Compare variance heatmaps
print("Low Variance Samples:")
fig = plot_variance_heatmap(samples_low_var[:20], title="Low Variance Samples")
plt.show()

print("High Variance Samples:")
fig = plot_variance_heatmap(samples_high_var[:20], title="High Variance Samples")
plt.show()

## 2. Determine Tolerance Multipliers

In [ ]:
# Calculate optimal tolerance as multiple of standard deviation
def calculate_tolerance(samples, multiplier=2.0):
    """
    Calculate tolerance values based on sample variance.
    
    Tolerance = multiplier * std
    
    Args:
        samples: List of landmark sets
        multiplier: Multiplier for std (2.0 = ~95% coverage)
    
    Returns:
        Dict with per-landmark tolerances
    """
    stats = analyze_variance(samples)
    std = stats['std']  # (n_landmarks, 3)
    
    # Position tolerance = sqrt(std_x^2 + std_y^2 + std_z^2) * multiplier
    position_std = np.sqrt(np.sum(std ** 2, axis=1))
    position_tolerance = position_std * multiplier
    
    # Per-coordinate tolerances
    x_tolerance = std[:, 0] * multiplier
    y_tolerance = std[:, 1] * multiplier
    z_tolerance = std[:, 2] * multiplier
    
    return {
        'position': float(np.mean(position_tolerance)),
        'position_per_landmark': position_tolerance,
        'x': float(np.mean(x_tolerance)),
        'y': float(np.mean(y_tolerance)),
        'z': float(np.mean(z_tolerance)),
    }

# Test different multipliers
multipliers = [1.0, 1.5, 2.0, 2.5, 3.0]

print("Tolerance values for medium variance samples:")
print("="*60)
print(f"{'Multiplier':>12} {'Position':>12} {'~Coverage':>12}")
print("-"*60)

for mult in multipliers:
    tol = calculate_tolerance(samples_med_var, multiplier=mult)
    # Approximate coverage based on normal distribution
    coverage = stats.norm.cdf(mult) - stats.norm.cdf(-mult)
    print(f"{mult:>12.1f} {tol['position']:>12.4f} {coverage*100:>11.1f}%")

In [ ]:
# Analyze per-landmark tolerances
tolerances_med = calculate_tolerance(samples_med_var, multiplier=2.0)

plt.figure(figsize=(12, 5))

landmark_tol = tolerances_med['position_per_landmark']
colors = ['purple'] + ['red']*4 + ['green']*4 + ['blue']*4 + ['orange']*4 + ['yellow']*4

plt.bar(range(21), landmark_tol, color=colors)
plt.axhline(np.mean(landmark_tol), color='red', linestyle='--', label=f'Mean: {np.mean(landmark_tol):.4f}')
plt.xlabel('Landmark Index')
plt.ylabel('Tolerance (2σ)')
plt.title('Per-Landmark Tolerance (2σ coverage)')
plt.legend()

# Add landmark names
for i, name in LANDMARK_NAMES.items():
    if i in [WRIST, THUMB_TIP, INDEX_FINGER_TIP, MIDDLE_FINGER_TIP, PINKY_TIP]:
        plt.annotate(name.replace('_', '\n'), (i, landmark_tol[i]), 
                     ha='center', va='bottom', fontsize=7, rotation=45)

plt.tight_layout()
plt.show()

print(f"\nRecommended global position tolerance: {np.mean(landmark_tol):.4f}")
print(f"Fingertip tolerance (higher variance): {np.mean([landmark_tol[i] for i in FINGERTIP_INDICES]):.4f}")

In [ ]:
# Compare tolerances across variance levels
fig, ax = plt.subplots(figsize=(10, 6))

x = np.arange(21)
width = 0.25

tol_low = calculate_tolerance(samples_low_var, 2.0)['position_per_landmark']
tol_med = calculate_tolerance(samples_med_var, 2.0)['position_per_landmark']
tol_high = calculate_tolerance(samples_high_var, 2.0)['position_per_landmark']

ax.bar(x - width, tol_low, width, label='Low Variance', alpha=0.8)
ax.bar(x, tol_med, width, label='Medium Variance', alpha=0.8)
ax.bar(x + width, tol_high, width, label='High Variance', alpha=0.8)

ax.set_xlabel('Landmark Index')
ax.set_ylabel('Tolerance (2σ)')
ax.set_title('Tolerance Comparison by Variance Level')
ax.legend()

plt.tight_layout()
plt.show()

print("\nSummary of recommended tolerances:")
print(f"  Low variance scenarios: {np.mean(tol_low):.4f}")
print(f"  Medium variance scenarios: {np.mean(tol_med):.4f}")
print(f"  High variance scenarios: {np.mean(tol_high):.4f}")

## 3. Visualize Tolerance Regions

In [ ]:
# Calculate canonical (mean) pose
def calculate_canonical(samples):
    """Calculate mean pose from samples."""
    coords_list = []
    for s in samples:
        x, y, z = landmarks_to_arrays(s)
        coords_list.append(np.stack([x, y, z], axis=1))
    
    stacked = np.stack(coords_list, axis=0)
    mean = np.mean(stacked, axis=0)
    
    return [Point3D(x=mean[i, 0], y=mean[i, 1], z=mean[i, 2]) for i in range(mean.shape[0])]

canonical = calculate_canonical(samples_med_var)
tolerances = calculate_tolerance(samples_med_var, multiplier=2.0)

print(f"Calculated canonical pose with {len(canonical)} landmarks")
print(f"Global tolerance: {tolerances['position']:.4f}")

In [ ]:
# Visualize tolerance regions
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for ax, view in zip(axes, ['front', 'side', 'top']):
    plot_tolerance_region(
        canonical,
        {'position': tolerances['position']},
        ax=ax,
        title=f'Tolerance Region ({view.title()} View)',
        view=view
    )

plt.tight_layout()
plt.show()

In [ ]:
# Overlay samples with tolerance regions
fig, ax = plt.subplots(figsize=(10, 10))

# Plot tolerance circles
plot_tolerance_region(
    canonical,
    {'position': tolerances['position']},
    ax=ax,
    title='Samples within Tolerance Region',
    view='front'
)

# Overlay sample points
for sample in samples_med_var[:10]:
    x, y, z = landmarks_to_arrays(sample)
    ax.scatter(x, y, c='red', s=10, alpha=0.3)

plt.show()

In [ ]:
# Show distance distribution with tolerance threshold
fig = plot_distance_distribution(
    samples_med_var,
    reference=canonical,
    title='Distance from Canonical Pose'
)

# Add tolerance threshold line
plt.figure(figsize=(10, 5))
distances = analysis['medium']['distances'].flatten()
plt.hist(distances, bins=50, alpha=0.7, edgecolor='black')
plt.axvline(tolerances['position'], color='red', linestyle='--', linewidth=2,
            label=f'Tolerance (2σ): {tolerances["position"]:.4f}')
plt.axvline(np.percentile(distances, 95), color='orange', linestyle=':', linewidth=2,
            label=f'95th percentile: {np.percentile(distances, 95):.4f}')
plt.xlabel('Distance from Canonical')
plt.ylabel('Frequency')
plt.title('Distance Distribution with Tolerance Threshold')
plt.legend()
plt.show()

# Calculate what percentage falls within tolerance
within_tol = np.mean(distances <= tolerances['position']) * 100
print(f"Percentage within 2σ tolerance: {within_tol:.1f}%")

## 4. Test Matching with Different Thresholds

In [ ]:
# Simple matching function
def match_pose(test_landmarks, canonical, tolerance):
    """
    Check if test landmarks match canonical within tolerance.
    
    Returns:
        (is_match, score, per_landmark_distances)
    """
    test_x, test_y, test_z = landmarks_to_arrays(test_landmarks)
    can_x, can_y, can_z = landmarks_to_arrays(canonical)
    
    # Calculate distances
    distances = np.sqrt(
        (test_x - can_x)**2 +
        (test_y - can_y)**2 +
        (test_z - can_z)**2
    )
    
    # Check if all within tolerance
    within_tolerance = distances <= tolerance
    is_match = np.all(within_tolerance)
    
    # Calculate match score (percentage of landmarks within tolerance)
    score = np.mean(within_tolerance)
    
    return is_match, score, distances

print("Matching function defined")

In [ ]:
# Test matching on known samples
print("Testing matching on medium variance samples:")
print("="*60)

test_tolerances = [0.02, 0.04, 0.06, 0.08, 0.10, 0.15]

for tol in test_tolerances:
    match_count = 0
    total_score = 0
    
    for sample in samples_med_var:
        is_match, score, _ = match_pose(sample, canonical, tol)
        if is_match:
            match_count += 1
        total_score += score
    
    match_rate = match_count / len(samples_med_var) * 100
    avg_score = total_score / len(samples_med_var) * 100
    
    print(f"Tolerance {tol:.2f}: Match rate = {match_rate:.1f}%, Avg score = {avg_score:.1f}%")

In [ ]:
# Test with non-matching poses (different sign)
# Create a clearly different pose
different_hand = create_sample_landmarks()
# Modify it to be different (e.g., different finger positions)
for i in [INDEX_FINGER_TIP, MIDDLE_FINGER_TIP]:
    different_hand[i] = Point3D(
        x=different_hand[i].x + 0.2,
        y=different_hand[i].y - 0.2,
        z=different_hand[i].z
    )

print("Testing with clearly different pose:")
print("="*60)

for tol in test_tolerances:
    is_match, score, distances = match_pose(different_hand, canonical, tol)
    status = "MATCH" if is_match else "NO MATCH"
    print(f"Tolerance {tol:.2f}: {status}, Score = {score*100:.1f}%, Max dist = {max(distances):.4f}")

In [ ]:
# Visualize match results
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Plot matching sample
is_match, score, distances = match_pose(samples_med_var[0], canonical, tolerances['position'])
ax = axes[0]
plot_hand_2d(canonical, ax=ax, title=f'Match: Score={score*100:.0f}%', view='front')

# Overlay test sample
test_x, test_y, _ = landmarks_to_arrays(samples_med_var[0])
colors = ['green' if d <= tolerances['position'] else 'red' for d in distances]
ax.scatter(test_x, test_y, c=colors, s=100, marker='x', linewidths=2)

# Plot non-matching sample
is_match, score, distances = match_pose(different_hand, canonical, tolerances['position'])
ax = axes[1]
plot_hand_2d(canonical, ax=ax, title=f'No Match: Score={score*100:.0f}%', view='front')

# Overlay test sample
test_x, test_y, _ = landmarks_to_arrays(different_hand)
colors = ['green' if d <= tolerances['position'] else 'red' for d in distances]
ax.scatter(test_x, test_y, c=colors, s=100, marker='x', linewidths=2)

plt.tight_layout()
plt.show()

print("Green X = within tolerance, Red X = outside tolerance")

In [ ]:
# Find optimal tolerance threshold
def evaluate_tolerance(tolerance, positive_samples, negative_samples, canonical):
    """
    Evaluate tolerance by calculating true positive and false positive rates.
    """
    # True positives (matching samples correctly matched)
    tp = sum(1 for s in positive_samples if match_pose(s, canonical, tolerance)[0])
    tp_rate = tp / len(positive_samples)
    
    # False positives (different samples incorrectly matched)
    fp = sum(1 for s in negative_samples if match_pose(s, canonical, tolerance)[0])
    fp_rate = fp / len(negative_samples)
    
    return tp_rate, fp_rate

# Create negative samples (different poses)
negative_samples = []
for i in range(20):
    diff = create_sample_landmarks()
    # Apply random larger displacement
    x, y, z = landmarks_to_arrays(diff)
    offset = np.random.uniform(-0.3, 0.3, (21, 3))
    for j in range(21):
        diff[j] = Point3D(x=x[j]+offset[j, 0], y=y[j]+offset[j, 1], z=z[j]+offset[j, 2])
    negative_samples.append(diff)

# Evaluate different thresholds
test_tols = np.linspace(0.01, 0.20, 20)
results = []

for tol in test_tols:
    tp_rate, fp_rate = evaluate_tolerance(tol, samples_med_var, negative_samples, canonical)
    results.append({'tolerance': tol, 'tp_rate': tp_rate, 'fp_rate': fp_rate})

results_df = pd.DataFrame(results)

# Plot ROC-like curve
plt.figure(figsize=(10, 6))
plt.plot(results_df['tolerance'], results_df['tp_rate'], 'g-', label='True Positive Rate', linewidth=2)
plt.plot(results_df['tolerance'], results_df['fp_rate'], 'r-', label='False Positive Rate', linewidth=2)
plt.axvline(tolerances['position'], color='blue', linestyle='--', label=f'2σ tolerance: {tolerances["position"]:.3f}')
plt.xlabel('Tolerance Threshold')
plt.ylabel('Rate')
plt.title('Matching Performance vs Tolerance')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# Find optimal point (maximize TP - FP)
results_df['score'] = results_df['tp_rate'] - results_df['fp_rate']
optimal = results_df.loc[results_df['score'].idxmax()]
print(f"\nOptimal tolerance: {optimal['tolerance']:.4f}")
print(f"  True positive rate: {optimal['tp_rate']*100:.1f}%")
print(f"  False positive rate: {optimal['fp_rate']*100:.1f}%")

In [ ]:
# Summary and recommendations
print("="*60)
print("TOLERANCE RECOMMENDATIONS")
print("="*60)
print(f"""
Based on analysis of sample data:

1. POSITION TOLERANCE:
   - Conservative (high precision): {np.mean(tol_low):.4f}
   - Balanced (recommended): {np.mean(tol_med):.4f}
   - Lenient (high recall): {np.mean(tol_high):.4f}

2. MULTIPLIER:
   - Use 2.0× standard deviation for ~95% coverage
   - Use 2.5× for more lenient matching
   - Use 1.5× for stricter matching

3. PER-LANDMARK TOLERANCE:
   - Fingertips: ~1.5× global tolerance (higher variance)
   - Palm landmarks: ~0.8× global tolerance (more stable)
   - Wrist: ~0.5× global tolerance (anchor point)

4. MATCHING STRATEGY:
   - Require all landmarks within tolerance for strict match
   - Use score threshold (e.g., >85%) for lenient match
   - Weight palm landmarks higher for stability
""")

## Next Steps

- **04_angle_calculations.ipynb** - Verify angle calculations
- **05_quality_metrics.ipynb** - Develop quality scoring